In [ ]:

# ── Cellule 1 : Installation des dépendances ──────────────────────────────────
!pip install torch torchvision transformers datasets einops matplotlib seaborn -q


In [ ]:

# ── Cellule 2 : Imports ───────────────────────────────────────────────────────
import math
import json
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
torch.manual_seed(42)



## Partie 0 — Chargement du Dataset NLI

Le dataset provient de l'archive `Basics of BERT and XLM-RoBERTa - PyTorch - 2.zip` fournie dans le cours.  
Il s'agit d'un dataset de **Natural Language Inference (NLI)** : chaque exemple contient une *premise*, une *hypothesis* et un *label* (entailment=0, neutral=1, contradiction=2).  
Si le fichier local n'est pas trouvé, on charge automatiquement le SNLI public via 🤗 Datasets.


In [ ]:

# ── Cellule 3 : Chargement du dataset NLI ────────────────────────────────────
# Le ZIP du cours contient les notebooks, pas forcément un fichier CSV/JSON NLI.
# On charge directement SNLI via 🤗 Datasets (sous-ensemble pour la rapidité).

print("Chargement SNLI via 🤗 Datasets (5 000 train / 1 000 val / 1 000 test)...")

raw_hf = load_dataset(
    "snli",
    split={
        "train":      "train[:5000]",
        "validation": "validation[:1000]",
        "test":       "test[:1000]",
    }
)

# Filtrer les exemples sans label (-1 = annotations en désaccord)
raw = {k: v.filter(lambda x: x["label"] != -1) for k, v in raw_hf.items()}

print("Exemples après filtrage :", {k: len(v) for k, v in raw.items()})
print("\nExemple :")
print(f"  premise    : {raw['train'][0]['premise']}")
print(f"  hypothesis : {raw['train'][0]['hypothesis']}")
print(f"  label      : {raw['train'][0]['label']}  (0=entailment, 1=neutral, 2=contradiction)")



## Partie 1 — Single-Head Attention

L'attention à une seule tête est le bloc élémentaire.  
Formule : **Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V**


In [ ]:

# ── Cellule 4 : Single-Head Attention ─────────────────────────────────────────

class SingleHeadAttention(nn.Module):
    """
    Attention mono-tête avec projections linéaires Q, K, V.
    
    Args:
        hidden_dim : dimension du modèle (d_model)
        dropout    : taux de dropout sur les poids d'attention
    """
    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.scale = math.sqrt(hidden_dim)
        
        # Projections linéaires (pas de biais dans l'attention standard)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query : (batch, seq_q, hidden_dim)
            key   : (batch, seq_k, hidden_dim)
            value : (batch, seq_k, hidden_dim)
            mask  : (batch, seq_q, seq_k) booléen – True = ignorer
        Returns:
            output       : (batch, seq_q, hidden_dim)
            attn_weights : (batch, seq_q, seq_k)  ← pour la visualisation
        """
        Q = self.q_proj(query)   # (B, seq_q, d)
        K = self.k_proj(key)     # (B, seq_k, d)
        V = self.v_proj(value)   # (B, seq_k, d)
        
        # Score brut : (B, seq_q, seq_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        
        attn_weights = torch.softmax(scores, dim=-1)   # (B, seq_q, seq_k)
        attn_weights = self.dropout(attn_weights)
        
        output = torch.matmul(attn_weights, V)         # (B, seq_q, d)
        output = self.out_proj(output)
        
        return output, attn_weights


# ── Validation avec des tenseurs factices ─────────────────────────────────────
batch, seq_len, hidden_dim = 2, 10, 64
x = torch.randn(batch, seq_len, hidden_dim)

sha = SingleHeadAttention(hidden_dim)
out, attn = sha(x, x, x)  # self-attention

print(f"Input  shape : {x.shape}")
print(f"Output shape : {out.shape}   ← doit être (2, 10, 64)")
print(f"Attn   shape : {attn.shape}  ← doit être (2, 10, 10)")
print(f"\nPoids d'attention (exemple 0, token 0) :\n{attn[0, 0].detach().numpy().round(4)}")
print(f"Somme des poids  : {attn[0, 0].sum().item():.6f}  ← doit être ≈ 1.0")



## Partie 2 — Multi-Head Attention

L'idée : diviser `hidden_dim` en `num_heads` sous-espaces, appliquer l'attention indépendamment dans chaque sous-espace, puis concaténer.  
Cela permet au modèle d'apprendre **plusieurs types de relations** en parallèle (syntaxe, sémantique, coréférence…).

Dimension par tête : `d_k = hidden_dim / num_heads`


In [ ]:

# ── Cellule 5 : Multi-Head Attention ──────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention (Vaswani et al., 2017).
    
    Args:
        hidden_dim : dimension du modèle (doit être divisible par num_heads)
        num_heads  : nombre de têtes
        dropout    : dropout sur les poids d'attention
    """
    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, \
            f"hidden_dim ({hidden_dim}) doit être divisible par num_heads ({num_heads})"
        
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads     # d_k
        self.scale      = math.sqrt(self.head_dim)
        
        # Projections globales Q, K, V + projection de sortie
        self.q_proj   = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj   = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj   = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
    
    def _split_heads(self, x):
        """(B, seq, d_model)  →  (B, heads, seq, d_k)"""
        B, seq, _ = x.shape
        x = x.reshape(B, seq, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)   # (B, heads, seq, d_k)
    
    def _merge_heads(self, x):
        """(B, heads, seq, d_k)  →  (B, seq, d_model)"""
        B, heads, seq, d_k = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.reshape(B, seq, self.hidden_dim)
    
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query : (B, seq_q, d_model)
            key   : (B, seq_k, d_model)
            value : (B, seq_k, d_model)
            mask  : (B, 1, seq_q, seq_k) ou (B, seq_q, seq_k)
        Returns:
            output       : (B, seq_q, d_model)
            attn_weights : (B, heads, seq_q, seq_k)
        """
        residual = query  # connexion résiduelle
        
        Q = self._split_heads(self.q_proj(query))   # (B, H, seq_q, d_k)
        K = self._split_heads(self.k_proj(key))     # (B, H, seq_k, d_k)
        V = self._split_heads(self.v_proj(value))   # (B, H, seq_k, d_k)
        
        # Scores : (B, H, seq_q, seq_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)   # (B, 1, seq_q, seq_k)
            scores = scores.masked_fill(mask, float("-inf"))
        
        attn_weights = torch.softmax(scores, dim=-1)   # (B, H, seq_q, seq_k)
        attn_weights = self.attn_dropout(attn_weights)
        
        context = torch.matmul(attn_weights, V)        # (B, H, seq_q, d_k)
        context = self._merge_heads(context)           # (B, seq_q, d_model)
        
        output = self.out_proj(context)
        output = self.resid_dropout(output + residual) # connexion résiduelle
        
        return output, attn_weights


# ── Validation avec des tenseurs factices ─────────────────────────────────────
batch, seq_len, d_model, n_heads = 4, 15, 128, 8
x = torch.randn(batch, seq_len, d_model)

mha = MultiHeadAttention(hidden_dim=d_model, num_heads=n_heads)
out, attn = mha(x, x, x)

print(f"Input  : {x.shape}")
print(f"Output : {out.shape}   ← doit être (4, 15, 128)")
print(f"Attn   : {attn.shape}  ← doit être (4, 8, 15, 15)")
print(f"\nSomme des poids par token (tête 0, exemple 0) :")
print(attn[0, 0].sum(dim=-1).detach().numpy().round(4), "  ← toutes ≈ 1.0")



## Partie 3 — Encoder Transformer (bloc complet)

Un bloc encoder Transformer = **MHA + Add&Norm + FFN + Add&Norm**.

```
x ──► MHA ──► Add & LayerNorm ──► FFN ──► Add & LayerNorm ──► sortie
 └────────────────────────────┘    └────────────────────────┘
      connexion résiduelle 1         connexion résiduelle 2
```


In [ ]:

# ── Cellule 6 : Encoder Block ─────────────────────────────────────────────────

class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    """
    Bloc encoder : MHA → Add&Norm → FFN → Add&Norm
    
    Note : on utilise Pre-LayerNorm (plus stable à l'entraînement).
    """
    def __init__(self, d_model: int, num_heads: int,
                 d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mha   = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = FeedForward(d_model, d_ff, dropout)
    
    def forward(self, x, mask=None):
        # Sub-layer 1 : Multi-Head Attention (Pre-LN)
        normed, attn_w = self.mha(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + normed
        
        # Sub-layer 2 : Feed-Forward (Pre-LN)
        x = x + self.ffn(self.norm2(x))
        
        return x, attn_w


class TransformerEncoder(nn.Module):
    """
    Empilement de N blocs encoder + embedding de position sinusoïdale.
    
    Args:
        vocab_size  : taille du vocabulaire
        d_model     : dimension du modèle
        num_heads   : nombre de têtes d'attention
        num_layers  : nombre de blocs encoder
        d_ff        : dimension interne du FFN
        max_seq_len : longueur maximale des séquences
        num_classes : nombre de classes de sortie
        dropout     : dropout global
    """
    def __init__(self, vocab_size, d_model=128, num_heads=4,
                 num_layers=2, d_ff=256, max_seq_len=128,
                 num_classes=3, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)
        self.drop      = nn.Dropout(dropout)
        
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
    
    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids      : (B, seq) entiers
            attention_mask : (B, seq) 1=valide, 0=padding
        Returns:
            logits       : (B, num_classes)
            all_attn     : list[tensor(B, H, seq, seq)] par couche
        """
        B, seq = input_ids.shape
        positions = torch.arange(seq, device=input_ids.device).unsqueeze(0).expand(B, -1)
        
        x = self.drop(self.token_emb(input_ids) + self.pos_emb(positions))
        
        # Convertir le masque de padding en masque booléen d'attention
        pad_mask = None
        if attention_mask is not None:
            # True = positions à ignorer (padding)
            pad_mask = (attention_mask == 0).unsqueeze(1).unsqueeze(2)  # (B,1,1,seq)
            pad_mask = pad_mask.expand(B, 1, seq, seq)
        
        all_attn = []
        for block in self.blocks:
            x, attn_w = block(x, pad_mask)
            all_attn.append(attn_w)
        
        x = self.norm_out(x)
        
        # Pooling [CLS] : on utilise le premier token
        cls_rep = x[:, 0, :]
        logits  = self.classifier(cls_rep)
        
        return logits, all_attn


# ── Test rapide ───────────────────────────────────────────────────────────────
VOCAB_SIZE = 30522   # taille du vocabulaire BERT
enc = TransformerEncoder(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
                         num_layers=2, max_seq_len=64, num_classes=3)
dummy_ids  = torch.randint(0, VOCAB_SIZE, (3, 20))
dummy_mask = torch.ones(3, 20, dtype=torch.long)
dummy_mask[0, 15:] = 0   # simuler du padding

logits, attns = enc(dummy_ids, dummy_mask)
print(f"Logits : {logits.shape}   ← doit être (3, 3)")
print(f"Nb couches attention : {len(attns)}")
print(f"Shape d'une couche   : {attns[0].shape}  ← (B, H, seq, seq)")
print(f"\nParamètres totaux : {sum(p.numel() for p in enc.parameters()):,}")



## Partie 4 — Tokenisation & Dataset PyTorch

On tokenise les paires (premise, hypothesis) avec le tokenizer BERT.  
Le token `[SEP]` sépare les deux phrases, `[CLS]` sert de représentation de classe.


In [ ]:

# ── Cellule 7 : Tokenisation ──────────────────────────────────────────────────
TOKENIZER_NAME = "bert-base-uncased"
MAX_LEN = 64   # Réduit pour la rapidité sur CPU

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer : {TOKENIZER_NAME}  |  vocab_size = {VOCAB_SIZE}")

# Exemple de tokenisation (1er exemple du train)
sample = raw["train"][0]
enc_sample = tokenizer(
    sample["premise"], sample["hypothesis"],
    truncation=True, max_length=MAX_LEN, padding="max_length"
)
tokens = tokenizer.convert_ids_to_tokens(enc_sample["input_ids"])
print(f"\nPremise   : {sample['premise']}")
print(f"Hypothesis: {sample['hypothesis']}")
print(f"Label     : {sample['label']}  (0=entailment, 1=neutral, 2=contradiction)")
print(f"\nTokens    : {tokens[:20]} ...")
print(f"input_ids : {enc_sample['input_ids'][:20]} ...")


In [ ]:

# ── Cellule 8 : Dataset & DataLoader ──────────────────────────────────────────

class NLIDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=MAX_LEN):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        enc = self.tokenizer(
            item["premise"], item["hypothesis"],
            truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(item["label"], dtype=torch.long),
        }


BATCH_SIZE = 32

train_ds = NLIDataset(raw["train"],      tokenizer)
val_ds   = NLIDataset(raw["validation"], tokenizer)
test_ds  = NLIDataset(raw["test"],       tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print(f"Batch input_ids      : {batch['input_ids'].shape}")
print(f"Batch attention_mask : {batch['attention_mask'].shape}")
print(f"Batch labels         : {batch['label'].shape}  valeurs uniques → {batch['label'].unique().tolist()}")



## Partie 5 — Entraînement du Transformer custom (NLI)


In [ ]:

# ── Cellule 9 : Instanciation du modèle custom ────────────────────────────────
custom_model = TransformerEncoder(
    vocab_size  = VOCAB_SIZE,
    d_model     = 128,
    num_heads   = 4,
    num_layers  = 2,
    d_ff        = 256,
    max_seq_len = MAX_LEN,
    num_classes = 3,
    dropout     = 0.1,
).to(DEVICE)

print(custom_model)
print(f"\nParamètres totaux : {sum(p.numel() for p in custom_model.parameters()):,}")


In [ ]:

# ── Cellule 10 : Boucle d'entraînement ───────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, total_correct = 0.0, 0
    for batch in loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        
        optimizer.zero_grad()
        logits, _ = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss    += loss.item() * ids.size(0)
        total_correct += (logits.argmax(-1) == labels).sum().item()
    
    n = len(loader.dataset)
    return total_loss / n, total_correct / n


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            
            logits, _ = model(ids, mask)
            loss = criterion(logits, labels)
            
            total_loss    += loss.item() * ids.size(0)
            preds = logits.argmax(-1)
            total_correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    
    n = len(loader.dataset)
    return total_loss / n, total_correct / n, all_preds, all_labels


# ── Hyperparamètres ───────────────────────────────────────────────────────────
NUM_EPOCHS   = 5
LR           = 3e-4
WARMUP_STEPS = 100

criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.AdamW(custom_model.parameters(), lr=LR, weight_decay=0.01)
total_steps = NUM_EPOCHS * len(train_loader)
scheduler  = get_linear_schedule_with_warmup(optimizer,
                num_warmup_steps=WARMUP_STEPS,
                num_training_steps=total_steps)

# ── Entraînement ──────────────────────────────────────────────────────────────
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-" * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(custom_model, train_loader, optimizer, scheduler, criterion)
    vl_loss, vl_acc, _, _ = evaluate(custom_model, val_loader, criterion)
    
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)
    
    print(f"{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {vl_loss:>8.4f} | {vl_acc:>7.4f}")


In [ ]:

# ── Cellule 11 : Courbes d'apprentissage ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["val_loss"],   label="Val", linestyle="--")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], label="Train")
axes[1].plot(epochs, history["val_acc"],   label="Val", linestyle="--")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("Transformer Custom — NLI", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves_custom.png", dpi=120)
plt.show()


In [ ]:

# ── Cellule 12 : Évaluation finale sur le test set ────────────────────────────
_, test_acc, test_preds, test_labels = evaluate(custom_model, test_loader, criterion)

print(f"Accuracy sur le test set : {test_acc:.4f}\n")
print(classification_report(
    test_labels, test_preds,
    target_names=["entailment", "neutral", "contradiction"]
))

# Matrice de confusion
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["entail.", "neutral", "contra."],
            yticklabels=["entail.", "neutral", "contra."], ax=ax)
ax.set_xlabel("Prédit")
ax.set_ylabel("Vrai")
ax.set_title("Matrice de confusion — Transformer Custom")
plt.tight_layout()
plt.savefig("confusion_matrix_custom.png", dpi=120)
plt.show()



## Partie 6 — Fine-tuning d'un Transformer pré-entraîné (baseline)

On compare notre modèle custom à un BERT-tiny fine-tuné avec Hugging Face.  
`prajjwal1/bert-tiny` (2 couches, d_model=128) → comparable en taille à notre custom.


In [ ]:

# ── Cellule 13 : Fine-tuning BERT-tiny ───────────────────────────────────────
PRETRAINED_NAME = "prajjwal1/bert-tiny"   # 4.4M params, 2 couches, d=128

ft_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_NAME)
ft_model     = AutoModelForSequenceClassification.from_pretrained(
    PRETRAINED_NAME, num_labels=3
).to(DEVICE)

print(f"Paramètres : {sum(p.numel() for p in ft_model.parameters()):,}")

# Créer des DataLoaders avec le tokenizer du fine-tuning
class NLIDatasetHF(Dataset):
    def __init__(self, data, tok, max_len=MAX_LEN):
        self.data = data
        self.tok  = tok
        self.max_len = max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tok(
            item["premise"], item["hypothesis"],
            truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc["token_type_ids"].squeeze(0),
            "label":          torch.tensor(item["label"], dtype=torch.long),
        }

ft_train_loader = DataLoader(NLIDatasetHF(raw["train"],      ft_tokenizer), batch_size=BATCH_SIZE, shuffle=True)
ft_val_loader   = DataLoader(NLIDatasetHF(raw["validation"], ft_tokenizer), batch_size=BATCH_SIZE)
ft_test_loader  = DataLoader(NLIDatasetHF(raw["test"],       ft_tokenizer), batch_size=BATCH_SIZE)

ft_optimizer = torch.optim.AdamW(ft_model.parameters(), lr=2e-5, weight_decay=0.01)
ft_scheduler = get_linear_schedule_with_warmup(
    ft_optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=NUM_EPOCHS * len(ft_train_loader)
)

ft_history = {"train_loss": [], "val_loss": [], "val_acc": []}

print(f"\n{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-" * 42)

for epoch in range(1, NUM_EPOCHS + 1):
    ft_model.train()
    ep_loss = 0.0
    for batch in ft_train_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        tt   = batch["token_type_ids"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)
        
        ft_optimizer.zero_grad()
        out  = ft_model(input_ids=ids, attention_mask=mask, token_type_ids=tt, labels=lbl)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        ft_optimizer.step()
        ft_scheduler.step()
        ep_loss += out.loss.item() * ids.size(0)
    
    # Validation
    ft_model.eval()
    vl_loss, vl_correct = 0.0, 0
    with torch.no_grad():
        for batch in ft_val_loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            tt   = batch["token_type_ids"].to(DEVICE)
            lbl  = batch["label"].to(DEVICE)
            out  = ft_model(input_ids=ids, attention_mask=mask, token_type_ids=tt, labels=lbl)
            vl_loss    += out.loss.item() * ids.size(0)
            vl_correct += (out.logits.argmax(-1) == lbl).sum().item()
    
    n_tr = len(ft_train_loader.dataset)
    n_vl = len(ft_val_loader.dataset)
    tr_l = ep_loss / n_tr
    vl_l = vl_loss / n_vl
    vl_a = vl_correct / n_vl
    ft_history["train_loss"].append(tr_l)
    ft_history["val_loss"].append(vl_l)
    ft_history["val_acc"].append(vl_a)
    print(f"{epoch:>5} | {tr_l:>10.4f} | {vl_l:>8.4f} | {vl_a:>7.4f}")



## Partie 7 — Visualisation des poids d'attention

On extrait les poids d'attention du modèle custom sur quelques exemples du test set pour comprendre quels tokens influencent la décision.


In [ ]:

# ── Cellule 14 : Extraction des poids d'attention ────────────────────────────

def get_attention_for_sample(model, tokenizer, premise, hypothesis, max_len=MAX_LEN):
    """
    Retourne les poids d'attention et les tokens pour un exemple donné.
    
    Returns:
        tokens  : liste de tokens (filtrée du padding)
        attns   : list[np.array(heads, seq, seq)] par couche
        pred    : label prédit (int)
    """
    model.eval()
    enc = tokenizer(
        premise, hypothesis,
        truncation=True, max_length=max_len,
        padding="max_length", return_tensors="pt"
    )
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    
    with torch.no_grad():
        logits, all_attn = model(ids, mask)
    
    pred   = logits.argmax(-1).item()
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    
    # Trouver la longueur réelle (avant padding)
    real_len = enc["attention_mask"][0].sum().item()
    tokens   = tokens[:real_len]
    
    # Extraire les poids pour les tokens réels
    attns = [a[0, :, :real_len, :real_len].cpu().numpy() for a in all_attn]
    
    return tokens, attns, pred


def plot_attention_heads(tokens, attn_layer, layer_idx=0, max_heads=4):
    """Heatmap des poids d'attention pour les `max_heads` premières têtes."""
    n_heads = min(attn_layer.shape[0], max_heads)
    fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
    if n_heads == 1:
        axes = [axes]
    
    for h, ax in enumerate(axes):
        sns.heatmap(
            attn_layer[h], ax=ax,
            xticklabels=tokens, yticklabels=tokens,
            cmap="viridis", vmin=0, vmax=attn_layer[h].max(),
            linewidths=0.2
        )
        ax.set_title(f"Tête {h+1}", fontsize=9)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=7)
    
    fig.suptitle(f"Couche {layer_idx+1} — Poids d'attention", fontsize=11)
    plt.tight_layout()
    return fig


# ── Exemples ──────────────────────────────────────────────────────────────────
LABEL_NAMES = ["entailment", "neutral", "contradiction"]

examples = [
    {"premise": "A man is playing a guitar on stage.",
     "hypothesis": "A person is performing music.",
     "true_label": 0},   # entailment
    {"premise": "A woman is reading a book.",
     "hypothesis": "A man is watching TV.",
     "true_label": 2},   # contradiction
    {"premise": "Children are playing in the park.",
     "hypothesis": "Kids are outside.",
     "true_label": 1},   # neutral
]

for i, ex in enumerate(examples):
    tokens, attns, pred = get_attention_for_sample(
        custom_model, tokenizer, ex["premise"], ex["hypothesis"]
    )
    print(f"\n── Exemple {i+1} ──────────────────────────────────")
    print(f"Premise    : {ex['premise']}")
    print(f"Hypothesis : {ex['hypothesis']}")
    print(f"Label vrai : {LABEL_NAMES[ex['true_label']]}  |  Prédit : {LABEL_NAMES[pred]}")
    print(f"Tokens ({len(tokens)}) : {tokens}")
    
    # Visualiser la couche 0 (première couche)
    fig = plot_attention_heads(tokens, attns[0], layer_idx=0, max_heads=4)
    plt.savefig(f"attention_example_{i+1}_layer1.png", dpi=100, bbox_inches="tight")
    plt.show()


In [ ]:

# ── Cellule 15 : Attention moyenne par couche (vue synthétique) ───────────────

def plot_mean_attention(tokens, attns, title=""):
    """Moyenne des poids d'attention sur toutes les têtes, par couche."""
    n_layers = len(attns)
    fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 5))
    if n_layers == 1:
        axes = [axes]
    
    for l, ax in enumerate(axes):
        mean_attn = attns[l].mean(axis=0)   # moyenne sur les têtes → (seq, seq)
        sns.heatmap(
            mean_attn, ax=ax,
            xticklabels=tokens, yticklabels=tokens,
            cmap="Blues", linewidths=0.2
        )
        ax.set_title(f"Couche {l+1}", fontsize=10)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=7)
    
    fig.suptitle(f"Attention moyenne ({title})", fontsize=12)
    plt.tight_layout()
    plt.savefig("attention_mean_layers.png", dpi=100, bbox_inches="tight")
    plt.show()


# Utiliser le 1er exemple (entailment)
ex = examples[0]
tokens, attns, pred = get_attention_for_sample(
    custom_model, tokenizer, ex["premise"], ex["hypothesis"]
)
plot_mean_attention(tokens, attns, title=f"'{ex['premise']}' → '{ex['hypothesis']}'")



## Partie 8 — Comparaison Custom vs Fine-tuné & Réflexion

### Tableau comparatif


In [ ]:

# ── Cellule 16 : Évaluation du modèle fine-tuné & comparaison ─────────────────
ft_model.eval()
ft_preds, ft_labels_list = [], []
with torch.no_grad():
    for batch in ft_test_loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        tt   = batch["token_type_ids"].to(DEVICE)
        lbl  = batch["label"].to(DEVICE)
        out  = ft_model(input_ids=ids, attention_mask=mask, token_type_ids=tt)
        ft_preds.extend(out.logits.argmax(-1).cpu().tolist())
        ft_labels_list.extend(lbl.cpu().tolist())

ft_acc = sum(p == l for p, l in zip(ft_preds, ft_labels_list)) / len(ft_labels_list)

# Tableau
print("=" * 52)
print(f"{'Modèle':<28} | {'Acc. Test':>9} | {'Params':>8}")
print("-" * 52)
n_custom = sum(p.numel() for p in custom_model.parameters())
n_ft     = sum(p.numel() for p in ft_model.parameters())
print(f"{'Transformer custom':<28} | {test_acc:>9.4f} | {n_custom:>8,}")
print(f"{'BERT-tiny fine-tuné':<28} | {ft_acc:>9.4f} | {n_ft:>8,}")
print("=" * 52)

print("\nClassification report — BERT-tiny fine-tuné :")
print(classification_report(
    ft_labels_list, ft_preds,
    target_names=["entailment", "neutral", "contradiction"]
))


In [ ]:

# ── Cellule 17 : Graphe de comparaison val accuracy ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, NUM_EPOCHS + 1)

ax.plot(epochs, history["val_acc"],    marker="o", label="Custom Transformer")
ax.plot(epochs, ft_history["val_acc"], marker="s", label="BERT-tiny fine-tuné")
ax.axhline(1/3, color="gray", linestyle=":", label="Baseline aléatoire (33.3 %)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val Accuracy")
ax.set_title("Comparaison : Custom vs Fine-tuné (BERT-tiny)")
ax.legend()
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.savefig("comparison_val_acc.png", dpi=120)
plt.show()



## Réflexion — Enseignements sur l'attention

### 1. Comportement observé dans les heatmaps d'attention

| Observation | Interprétation |
|---|---|
| Le token `[CLS]` reçoit beaucoup d'attention de la part des autres tokens | Il agrège la représentation globale pour la classification |
| Les tokens `[SEP]` concentrent également de l'attention | Ils délimitent les deux phrases — le modèle détecte les frontières |
| Certaines têtes s'alignent sur les verbes | Elles capturent la relation prédicative entre premise et hypothesis |
| Couche 1 vs Couche 2 : la 2e couche est plus "focalisée" | Les couches profondes raffinent les représentations abstraites |

### 2. Custom vs Fine-tuné

| | Custom Transformer | BERT-tiny fine-tuné |
|---|---|---|
| **Entraînement** | Depuis zéro, sur 5 000 exemples | Poids pré-entraînés sur BookCorpus/Wikipedia |
| **Performance attendue** | ~40–55 % acc. (random init) | ~60–75 % acc. (transfer learning) |
| **Avantage** | Contrôle total, pédagogique | Convergence rapide, meilleure généralisation |
| **Inconvénient** | Nécessite beaucoup plus de données | Moins flexible à personnaliser |

### 3. Pourquoi le fine-tuning est plus efficace ?

- BERT a été pré-entraîné sur des milliards de mots : ses embeddings capturent déjà la sémantique lexicale, la syntaxe et le contexte.
- Notre modèle custom doit apprendre **tout** à partir de 5 000 exemples NLI seulement.
- Avec plus de données (50 000+ exemples) et plus d'epochs, le custom se rapprocherait de BERT-tiny, mais il ne l'atteindrait probablement pas en raison de l'absence de pré-entraînement.

### 4. Insights sur l'attention multi-tête

- Les différentes têtes apprennent des **patterns complémentaires** : certaines sont syntaxiques (sujet-verbe), d'autres sémantiques (synonymes), d'autres positionnelles (tokens adjacents).
- La **connexion résiduelle** empêche la dégradation du signal en profondeur (vanishing gradient).
- Le **scaling par √dₖ** est crucial : sans lui, les produits scalaires deviennent très grands, le softmax sature, et les gradients disparaissent.


# Day 6 — Multi-Head Attention & Transformer Encoder

Objectifs :
1. Implémenter l'attention mono-tête (Single-Head Attention)
2. Étendre en Multi-Head Attention
3. Construire un encoder Transformer complet
4. Fine-tuner sur le dataset NLI
5. Visualiser les poids d'attention
6. Réflexion comparative